# recs_013 — D1 heuristic ranker on `two_tower_v1` pools

**Goal:** Rerank the frozen **two-tower retrieval pool** (top-100) with a hand-built blend:

```text
score = alpha * norm(retrieval_score) + (1 - alpha) * norm(train_popularity)
```

Then take top-10 for ranking metrics (`k_final=10`).

**Inputs (no re-scoring the catalog):**
- Cached pools: `eval_offline_examples.jsonl` rows where `method == two_tower_v1`
- Popularity prior: train positive-review counts from `prepare_eval_inputs_from_cache`

**Compare:** bare two-tower rank order vs best D1 vs oracle ceiling (same pool).

Plan: [`docs/ranker_exploration_plan.md`](../../docs/ranker_exploration_plan.md) · Concepts: [`recs_013_ranker_approaches_learn.ipynb`](recs_013_ranker_approaches_learn.ipynb)

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from steam_review_ml.evaluation.retrieval_offline_eval import (
    _oracle_ranked_indices_from_retrieved,
    _rank_rows,
    average_precision_at_k,
    hit_rate_at_k,
    mrr,
    ndcg_at_k,
    prepare_eval_inputs_from_cache,
)

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())

# --- config ---
EVAL_RUN = "latest"  # or archive folder name under offline_eval/runs/
POOL_METHOD = "two_tower_v1"
METHOD_D1 = "two_tower_v1_heuristic_pop"
K_FINAL = 10
K_RETRIEVAL = 100
MIN_REVIEW_CHARS = 30
EXAMPLES_PARQUET = REPO_ROOT / "artifacts/recs/eval_cache/val_dev_12k_v1/eval_examples.parquet"
ARTIFACT_DIR = REPO_ROOT / "artifacts/recs"

RUNS_ROOT = REPO_ROOT / "artifacts/recs/offline_eval/runs"
EVAL_DIR = RUNS_ROOT / "latest" if EVAL_RUN == "latest" else RUNS_ROOT / EVAL_RUN
JSONL_PATH = EVAL_DIR / "eval_offline_examples.jsonl"

if not JSONL_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {JSONL_PATH}. Run:\n"
        "  python scripts/recs_job_eval_retrieval.py configs/recs_job_eval_retrieval.json "
        "  --examples-parquet artifacts/recs/eval_cache/val_dev_12k_v1/eval_examples.parquet"
    )

print(f"EVAL_DIR={EVAL_DIR}")
print(f"POOL_METHOD={POOL_METHOD}  K_RETRIEVAL={K_RETRIEVAL}  K_FINAL={K_FINAL}")

In [ ]:
def load_two_tower_pools(jsonl_path: Path, *, method: str) -> list[dict]:
    """Load cached retrieval pools for one method from eval_offline_examples.jsonl."""
    pools: list[dict] = []
    with jsonl_path.open(encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            if row.get("method") != method:
                continue
            pools.append(row)
    if not pools:
        raise RuntimeError(f"No rows for method={method!r} in {jsonl_path}")
    return pools


def minmax_norm(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    lo, hi = float(x.min()), float(x.max())
    if hi - lo <= 1e-12:
        return np.zeros_like(x)
    return (x - lo) / (hi - lo)


def d1_pool_scores(
    pool_app_ids: list[int],
    retrieval_scores: list[float],
    *,
    alpha: float,
    pop_row: np.ndarray,
    app_to_row: dict[int, int],
) -> np.ndarray:
    """Heuristic blend within the frozen pool only."""
    pops = np.asarray([float(pop_row[app_to_row[int(a)]]) for a in pool_app_ids], dtype=np.float64)
    retr = minmax_norm(np.asarray(retrieval_scores, dtype=np.float64))
    pop = minmax_norm(pops)
    return alpha * retr + (1.0 - alpha) * pop


def pool_scores_to_ranked_indices(
    pool_app_ids: list[int],
    pool_scores: np.ndarray,
    *,
    app_ids: np.ndarray,
    app_to_row: dict[int, int],
    k_final: int,
) -> np.ndarray:
    """Scatter pool scores onto catalog rows; rank globally; return top-k row indices."""
    full = np.full(len(app_ids), -np.inf, dtype=np.float64)
    for app_id, score in zip(pool_app_ids, pool_scores):
        full[int(app_to_row[int(app_id)])] = float(score)
    return _rank_rows(full)[:k_final]


pools = load_two_tower_pools(JSONL_PATH, method=POOL_METHOD)
print(f"Loaded {len(pools):,} cached {POOL_METHOD} pools from {JSONL_PATH.name}")

In [ ]:
inputs = prepare_eval_inputs_from_cache(
    repo_root=REPO_ROOT,
    split="val",
    min_review_chars=MIN_REVIEW_CHARS,
    examples_parquet=EXAMPLES_PARQUET,
    artifact_dir=ARTIFACT_DIR,
    verbose=True,
)

app_ids = inputs.app_ids
app_to_row = inputs.app_to_row
pop_row = inputs.pop_row
print(f"Catalog size: {len(app_ids):,} games")

## Alpha grid (tune on Slice A only)

Slice A = `n_eval_targets >= 2`. Pick `alpha` with best mean **NDCG@10** on that slice.

> This is a small hyperparameter search on val — keep it simple for D1.

In [ ]:
ALPHAS = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]


def mean_ndcg_for_alpha(alpha: float, *, slice_a_only: bool) -> float:
    vals: list[float] = []
    for row in pools:
        if slice_a_only and int(row["n_eval_targets"]) < 2:
            continue
        positives = set(int(x) for x in json.loads(row["validation_positive_app_ids_json"]))
        if not positives:
            continue
        pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
        ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
        if alpha >= 1.0 - 1e-12:
            blend = np.asarray(ret_sc, dtype=np.float64)
        elif alpha <= 1e-12:
            blend = d1_pool_scores(pool_apps, ret_sc, alpha=0.0, pop_row=pop_row, app_to_row=app_to_row)
        else:
            blend = d1_pool_scores(pool_apps, ret_sc, alpha=alpha, pop_row=pop_row, app_to_row=app_to_row)
        ranked = pool_scores_to_ranked_indices(
            pool_apps, blend, app_ids=app_ids, app_to_row=app_to_row, k_final=K_FINAL
        )
        vals.append(ndcg_at_k(ranked, positives, K_FINAL, app_ids))
    return float(np.mean(vals)) if vals else float("nan")


grid_rows = []
for alpha in ALPHAS:
    grid_rows.append(
        {
            "alpha": alpha,
            "NDCG@K_slice_a": mean_ndcg_for_alpha(alpha, slice_a_only=True),
            "NDCG@K_all": mean_ndcg_for_alpha(alpha, slice_a_only=False),
        }
    )

alpha_grid = pd.DataFrame(grid_rows).sort_values("NDCG@K_slice_a", ascending=False)
display(alpha_grid)

BEST_ALPHA = float(alpha_grid.iloc[0]["alpha"])
print(f"Best alpha on Slice A: {BEST_ALPHA}")

## Full metric comparison

| Method | Description |
|--------|-------------|
| `two_tower_v1` | Baseline — rank by retrieval score within pool |
| `two_tower_v1_heuristic_pop` | D1 with tuned `alpha` |
| `two_tower_v1_oracle` | Positives first within pool (ranking ceiling) |

In [ ]:
def per_example_metrics(row: dict, *, alpha: float | None, oracle: bool = False) -> dict:
    positives = set(int(x) for x in json.loads(row["validation_positive_app_ids_json"]))
    pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
    ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]

    # full catalog row indices for retrieved pool (retrieval order preserved)
    retrieved_rows = np.asarray([app_to_row[a] for a in pool_apps], dtype=np.int64)

    if oracle:
        ranked = _oracle_ranked_indices_from_retrieved(retrieved_rows, positives, app_ids)[:K_FINAL]
        method = f"{POOL_METHOD}_oracle"
    elif alpha is None:
        ranked = pool_scores_to_ranked_indices(
            pool_apps, np.asarray(ret_sc, dtype=np.float64),
            app_ids=app_ids, app_to_row=app_to_row, k_final=K_FINAL,
        )
        method = POOL_METHOD
    else:
        blend = d1_pool_scores(pool_apps, ret_sc, alpha=alpha, pop_row=pop_row, app_to_row=app_to_row)
        ranked = pool_scores_to_ranked_indices(
            pool_apps, blend, app_ids=app_ids, app_to_row=app_to_row, k_final=K_FINAL,
        )
        method = METHOD_D1

    return {
        "method": method,
        "ex_idx": int(row["ex_idx"]),
        "slice_name": row["slice_name"],
        "n_eval_targets": int(row["n_eval_targets"]),
        "Hit@K": hit_rate_at_k(ranked, positives, K_FINAL, app_ids),
        "MAP@K": average_precision_at_k(ranked, positives, K_FINAL, app_ids),
        "NDCG@K": ndcg_at_k(ranked, positives, K_FINAL, app_ids),
        "MRR": mrr(ranked, positives, app_ids),
    }


metric_rows: list[dict] = []
for row in pools:
    positives = set(int(x) for x in json.loads(row["validation_positive_app_ids_json"]))
    if not positives:
        continue
    metric_rows.append(per_example_metrics(row, alpha=None))
    metric_rows.append(per_example_metrics(row, alpha=BEST_ALPHA))
    metric_rows.append(per_example_metrics(row, alpha=None, oracle=True))

df_ex = pd.DataFrame(metric_rows)


def aggregate(df: pd.DataFrame) -> pd.DataFrame:
    cols = ["Hit@K", "MAP@K", "NDCG@K", "MRR"]
    overall = df.groupby("method", as_index=False)[cols].mean()
    return overall.sort_values("NDCG@K", ascending=False)


display(Markdown("### Ranking overall (all examples)"))
display(aggregate(df_ex))

display(Markdown("### Ranking by slice (Slice A primary)"))
slice_tbl = (
    df_ex.groupby(["slice_name", "method"], as_index=False)[["Hit@K", "NDCG@K", "MRR"]].mean()
    .sort_values(["slice_name", "NDCG@K"], ascending=[True, False])
)
display(slice_tbl)

In [ ]:
# Ranker headroom: oracle gap on Slice A for baseline vs D1
slice_a = df_ex[df_ex["slice_name"] == "slice_a_multi_target"]
summary = (
    slice_a.groupby("method")["NDCG@K"].mean()
    .rename("mean_NDCG@K_slice_a")
    .reset_index()
)
oracle_ndcg = float(summary.loc[summary["method"] == f"{POOL_METHOD}_oracle", "mean_NDCG@K_slice_a"].iloc[0])
for method in [POOL_METHOD, METHOD_D1]:
    base = float(summary.loc[summary["method"] == method, "mean_NDCG@K_slice_a"].iloc[0])
    print(f"{method}: NDCG@10={base:.4f}  oracle_gap={oracle_ndcg - base:.4f}")

display(summary)

## Next steps

1. If D1 wins on Slice A, wire `two_tower_v1_heuristic_pop` into `recs_job_eval_retrieval.py` as a composite scorer.
2. Try the same D1 on **`raw`** pools (plan C1) for comparison.
3. If oracle gap remains large after D1, proceed to **D2** (pointwise) with the same cached pools.

**Note:** Tuning `alpha` on the same val examples you report is optimistic — for a stricter check, hold out a query subset or use train-split pools only for tuning.